# E2a：从静态神经场走向动态世界

NeRF 的第一步是学一个函数：给定空间坐标，交出密度和颜色。我们先拟合一颗彩色小球，再说明时间与动作怎样进入接口。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.spatial import TinyNeuralField, make_colored_sphere_samples
torch.manual_seed(0)


## 1. 坐标不是图片编号

训练样本是 `(x,y,z) → density,color`。同一个连续函数可以在没有见过的坐标上查询。

In [ ]:
coordinates, density, color = make_colored_sphere_samples(640, seed=0)
print('coordinates/density/color:', tuple(coordinates.shape), tuple(density.shape), tuple(color.shape))
assert coordinates.shape[1] == 3


## 2. 拟合最小神经场

密度说明射线在哪里遇到物体，颜色说明该处呈现什么。真正 NeRF 还会沿相机射线采样并做体渲染。

In [ ]:
field = TinyNeuralField()
opt = torch.optim.Adam(field.parameters(), lr=5e-3)
losses = []
for _ in range(80):
    opt.zero_grad(); predicted_density, predicted_color = field(coordinates); loss = torch.nn.functional.mse_loss(predicted_density, density) + torch.nn.functional.mse_loss(predicted_color, color); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('field loss:', round(losses[0], 3), '→', round(losses[-1], 3))
assert losses[-1] < losses[0]


## 3. 静态重建为什么还不是 4D 世界

当前函数只有 `(x,y,z)`。加入时间以后变成 `(x,y,z,t)`，只能描述场景随时间变化；若想回答‘换动作会怎样’，还要加入动作或控制历史。

In [ ]:
print('static field input:  (x,y,z)')
print('dynamic field input: (x,y,z,t)')
print('controlled world:     (x,y,z,t,action/history)')
print('本 smoke 只完成第一行，不把彩色球称为 4D 世界。')


## 4. NeRF、3DGS 与 Mesh 的选择

NeRF 用连续网络查询，适合高质量新视角；3DGS 用显式高斯快速渲染；Mesh 用三角形提供明确表面，便于编辑与碰撞。它们是空间表示，不自动包含动作动态。

In [ ]:
representations = {'NeRF': '连续场，查询慢但平滑', '3DGS': '显式高斯，渲染快', 'Mesh': '明确表面，方便碰撞'}
for name, purpose in representations.items(): print(f'{name:5s}: {purpose}')


## 小结

这份 Notebook 只交出神经场的最小坐标接口。PA1-E2a 才使用多视角 Lego 或项目内 moving-shapes，加入新视角、时间一致性与动作条件检查。